# Clean V2 RidgeLinUCB Evaluation with Binary and Ordinal Rewards

This notebook evaluates the active V2 feature sets using RidgeLinUCB. It preserves the original binary correctness reward and adds the ordinal bucket reward:

- exact bucket: `+1`
- adjacent bucket error: `0`
- severe low/high bucket error: `-1`

Final accuracy is still reported as the fraction of exactly correct dose buckets, so the binary and ordinal reward runs remain directly comparable.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path] + list(path.parents):
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not find repository root.')


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.bandits.linUCB import RidgeLinUCB
from src.evaluation.feature_set_benchmark import (
    FINAL_FEATURE_SETS,
    FINAL_RIDGE_GRID,
    REWARD_SCHEMES_TO_COMPARE,
    available_final_feature_sets,
    classwise_metrics,
    confusion_table,
    evaluate_feature_set_grid_for_rewards,
    list_feature_sets,
    load_v2_assets,
    plot_feature_set_accuracy,
    save_benchmark_outputs,
    static_policy_summary,
    summarize_error_metrics,
)

RESULTS_DIR = REPO_ROOT / 'results' / 'v2_feature_set_evaluation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Repo root:', REPO_ROOT)
print('Results dir:', RESULTS_DIR)

## 1. Load cleaned V2 assets

In [ ]:
df_v2, feature_sets, preprocess_output_dir = load_v2_assets(REPO_ROOT)
feature_set_names = available_final_feature_sets(feature_sets, FINAL_FEATURE_SETS)

print('V2 modeling table:', df_v2.shape)
print('V2 preprocessing output:', preprocess_output_dir)
print('Active feature sets:', feature_set_names)
print('Reward schemes:', REWARD_SCHEMES_TO_COMPARE)

list_feature_sets(feature_sets)

## 2. Static baseline checks

Static baselines do not learn online, but we still keep their exact-bucket accuracy and ordinal mean reward for reference.

In [ ]:
static_baselines = static_policy_summary(df_v2, feature_set_names)
static_baselines.to_csv(RESULTS_DIR / 'static_baseline_summary.csv', index=False)
static_baselines.sort_values(['feature_set', 'accuracy'], ascending=[True, False])

## 3. RidgeLinUCB final grid under both reward schemes

In [ ]:
SEEDS = range(5)
RIDGE_ALPHAS = FINAL_RIDGE_GRID['alphas']
RIDGE_LAMBDAS = FINAL_RIDGE_GRID['lambda_regs']

ridge_summary, ridge_results, ridge_scale_stats = evaluate_feature_set_grid_for_rewards(
    df=df_v2,
    feature_sets=feature_sets,
    linucb_cls=RidgeLinUCB,
    feature_set_names=feature_set_names,
    alphas=RIDGE_ALPHAS,
    lambda_regs=RIDGE_LAMBDAS,
    reward_schemes=REWARD_SCHEMES_TO_COMPARE,
    seeds=SEEDS,
    standardize=True,
    progress=False,
    verbose=True,
)

ridge_summary.head(20)

## 4. Save metrics

In [ ]:
ridge_error_summary = summarize_error_metrics(ridge_results)
ridge_classwise = classwise_metrics(ridge_results)
ridge_confusion_true = confusion_table(ridge_results, normalize='true')
ridge_confusion_counts = confusion_table(ridge_results, normalize=None)

ridge_summary_with_stage = ridge_summary.copy()
ridge_summary_with_stage['stage'] = 'binary_and_ordinal_reward_grid'

written = save_benchmark_outputs(
    RESULTS_DIR,
    combined_feature_set_leaderboard=ridge_summary_with_stage,
    ridge_feature_set_step_results=ridge_results,
    ridge_feature_set_scale_stats=ridge_scale_stats,
    ridge_feature_set_error_summary=ridge_error_summary,
    ridge_feature_set_classwise=ridge_classwise,
    ridge_feature_set_confusion_true_normalized=ridge_confusion_true,
    ridge_feature_set_confusion_counts=ridge_confusion_counts,
    static_baseline_summary=static_baselines,
)
written

## 5. Leaderboard and diagnostics

In [ ]:
ridge_summary.sort_values('final_accuracy_mean', ascending=False).head(20)

In [ ]:
ridge_error_summary.sort_values('accuracy_mean', ascending=False).head(20)

In [ ]:
best = ridge_summary.iloc[0]
best_filter = (
    (ridge_classwise['feature_set'] == best['feature_set'])
    & (ridge_classwise['algorithm'] == best['algorithm'])
    & (ridge_classwise['reward_scheme'] == best['reward_scheme'])
    & (ridge_classwise['alpha'] == best['alpha'])
    & (ridge_classwise['lambda_reg'] == best['lambda_reg'])
)
ridge_classwise.loc[best_filter].sort_values('class_id')

In [ ]:
fig, ax = plot_feature_set_accuracy(
    ridge_summary,
    top_n=20,
    title='Clean V2 RidgeLinUCB leaderboard: binary vs ordinal reward',
)
fig.savefig(RESULTS_DIR / 'clean_v2_ridge_reward_leaderboard.png', dpi=180, bbox_inches='tight')
plt.show()

## 6. Reward comparison checkpoint

Use this table to compare whether ordinal reward improves exact bucket accuracy, low/high recall, or severe-error rates. The `final_reward_mean` column is reward-scheme-specific; `final_accuracy_mean` remains exact bucket accuracy for both reward schemes.

In [ ]:
comparison_cols = [
    'reward_scheme', 'feature_set', 'algorithm', 'alpha', 'lambda_reg',
    'final_accuracy_mean', 'final_accuracy_ci95',
    'final_reward_mean', 'late_accuracy_mean', 'final_regret_mean',
]
ridge_summary[comparison_cols].sort_values('final_accuracy_mean', ascending=False).head(30)